# L5: 异步编程基础

> 理解 Python 异步编程，掌握 async/await

## 学习目标

完成本课程后，你将能够：

- 理解异步编程的核心概念和工作原理
- 掌握 `async/await` 语法的正确使用
- 熟练使用 `asyncio` 库进行并发编程
- 了解事件循环、协程、Task 的关系
- 在 Agent 开发中正确使用异步编程
- 避免常见的异步编程陷阱

In [1]:
# === 环境设置 ===
import sys
import os
import time
from pathlib import Path

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

项目路径: c:\Users\zying\Desktop\pro_agent\agent-learning-project
Python 版本: 3.14.4


## 5.1 什么是异步编程？

### 核心概念

**异步编程（Asynchronous Programming）** 是一种并发编程方式，允许程序在等待某些操作（如网络请求、文件读写）完成时，去执行其他任务。

### 生活中的类比

想象你在餐厅点餐：

**同步（阻塞）方式：**
```
你 → 点餐 → 等待（干坐着）→ 拿餐 → 吃完 → 下一道菜
     每次只能等一道菜做好，才能点下一道
     总时间 = 菜1时间 + 菜2时间 + 菜3时间
```

**异步（非阻塞）方式：**
```
你 → 同时点所有菜 → 厨师并行做菜 → 菜好了端上来
     在等餐时你可以做其他事（刷手机、聊天）
     总时间 = max(菜1时间, 菜2时间, 菜3时间)
```

### 为什么 Agent 开发需要异步？

在 Agent 应用中，异步编程至关重要，因为：

| 操作 | 耗时 | 如果不用异步... |
|------|------|-----------------|
| LLM API 调用 | 1-5秒 | 用户等待，无法处理其他请求 |
| 向量数据库检索 | 0.1-1秒 | 阻塞主流程 |
| 网络搜索 | 1-3秒 | 浪费等待时间 |
| 多工具并发调用 | 累加时间 | 用户体验极差 |

### 同步 vs 异步对比

**同步代码的问题：**
```python
# 同步方式：串行执行
def answer_question(question):
    # 步骤1：搜索资料（2秒）
    search_results = search_web(question)  # 阻塞等待
    
    # 步骤2：获取历史对话（1秒）
    history = get_history(user_id)  # 又阻塞等待
    
    # 步骤3：调用 LLM（3秒）
    answer = call_llm(question, history)  # 继续阻塞
    
    # 总耗时：2 + 1 + 3 = 6 秒
    return answer
```

**异步代码的优势：**
```python
# 异步方式：并发执行
async def answer_question(question):
    # 步骤1和2可以同时进行！
    results = await asyncio.gather(
        search_web(question),   # 2秒
        get_history(user_id),   # 1秒
    )
    search_results, history = results
    
    # 步骤3：调用 LLM（3秒）
    answer = await call_llm(question, history)
    
    # 总耗时：max(2, 1) + 3 = 4 秒（节省了33%的时间！）
    return answer
```

### 关键术语

| 术语 | 英文 | 解释 |
|------|------|------|
| **协程** | Coroutine | 可以暂停和恢复的函数，用 `async def` 定义 |
| **事件循环** | Event Loop | 管理和调度协程执行的核心机制 |
| **任务** | Task | 包装后的协程，可以被事件循环调度 |
| **Future** | Future | 表示一个尚未完成的操作的结果 |
| **await** | await | 暂停当前协程，等待另一个协程完成

### 同步 vs 异步对比

In [2]:
import time
import asyncio

# 同步版本
def sync_task(name, seconds):
    print(f"{name}: 开始")
    time.sleep(seconds)  # 阻塞等待
    print(f"{name}: 完成")
    return f"{name}的结果"

def sync_main():
    start = time.time()
    
    result1 = sync_task("任务1", 1)
    result2 = sync_task("任务2", 1)
    result3 = sync_task("任务3", 1)
    
    elapsed = time.time() - start
    print(f"\n同步版本耗时: {elapsed:.2f}秒")
    print(f"结果: {[result1, result2, result3]}")

# 异步版本
async def async_task(name, seconds):
    print(f"{name}: 开始")
    await asyncio.sleep(seconds)  # 非阻塞等待
    print(f"{name}: 完成")
    return f"{name}的结果"

async def async_main():
    start = time.time()
    
    # 并发执行三个任务
    results = await asyncio.gather(
        async_task("任务1", 1),
        async_task("任务2", 1),
        async_task("任务3", 1),
    )
    
    elapsed = time.time() - start
    print(f"\n异步版本耗时: {elapsed:.2f}秒")
    print(f"结果: {results}")

# 运行对比
print("=== 同步版本 ===")
sync_main()

print("\n=== 异步版本 ===")
await async_main()

=== 同步版本 ===
任务1: 开始
任务1: 完成
任务2: 开始
任务2: 完成
任务3: 开始
任务3: 完成

同步版本耗时: 3.00秒
结果: ['任务1的结果', '任务2的结果', '任务3的结果']

=== 异步版本 ===
任务1: 开始
任务2: 开始
任务3: 开始
任务1: 完成
任务2: 完成
任务3: 完成

异步版本耗时: 1.00秒
结果: ['任务1的结果', '任务2的结果', '任务3的结果']


## 5.2 async/await 关键字

### 核心语法

| 关键字 | 作用 | 使用位置 | 示例 |
|--------|------|----------|------|
| `async def` | 定义协程函数 | 函数定义行 | `async def my_func():` |
| `await` | 等待协程完成 | async 函数内部 | `result = await async_func()` |
| `async with` | 异步上下文管理器 | async 函数内部 | `async with resource:` |
| `async for` | 异步迭代 | async 函数内部 | `async for item in stream:` |

### 定义协程函数

```python
# ✅ 正确：定义协程函数
async def greet(name):
    return f"Hello, {name}!"

# ❌ 错误：普通函数内部不能使用 await
def bad_example():
    result = await greet("Alice")  # SyntaxError!
```

### 调用协程函数

**重要：调用协程函数不会立即执行代码！**

```python
async def say_hello():
    print("开始...")
    await asyncio.sleep(1)
    print("完成！")
    return "结果"

# 调用协程函数返回的是协程对象，不是实际结果
coro = say_hello()  
print(coro)  # <coroutine object say_hello at 0x...>

# 要获取结果，必须用 await（在 async 函数内）
result = await say_hello()  # 或 asyncio.run(say_hello())
print(result)  # 结果
```

### await 的作用

`await` 关键字会：
1. **暂停**当前协程的执行
2. **让出控制权**给事件循环
3. **等待**被调用的协程完成
4. **恢复**当前协程的执行

```python
import asyncio

async def task_a():
    print("A: 开始")
    await asyncio.sleep(1)  # 在这里暂停，让其他任务运行
    print("A: 完成")
    return "A的结果"

async def task_b():
    print("B: 开始")
    await asyncio.sleep(0.5)  # 更快完成
    print("B: 完成")
    return "B的结果"

async def main():
    # 串行执行（因为用了 await）
    result_a = await task_a()  # 必须等 A 完成
    result_b = await task_b()  # 才能执行 B
    print(f"结果: {result_a}, {result_b}")

# 运行结果：A完成后才执行B
```

### 语法规则总结

1. **async def** 定义的函数叫"协程函数"
2. 调用协程函数返回"协程对象"
3. 协程对象必须被 **await** 或 **事件循环** 调度才会执行
4. **await** 只能在 **async def** 函数内使用
5. **不能**在普通函数内使用 await

In [6]:
import asyncio

# 定义一个简单的协程
async def say_hello(name):
    await asyncio.sleep(1)  # 模拟耗时操作
    return f"Hello, {name}!"

# 调用协程（错误方式）
try:
    result = say_hello("Alice")  # 返回协程对象，不会执行
    print(f"结果类型: {type(result)}")
    print(f"结果: {result}")
except:
    pass

# 正确方式：使用 await
result = await say_hello("Bob")
print(f"\n使用 await: {result}")

C:\Users\zying\AppData\Local\Temp\ipykernel_20952\145965430.py:10: RuntimeWarning: coroutine 'say_hello' was never awaited
  result = say_hello("Alice")  # 返回协程对象，不会执行


结果类型: <class 'coroutine'>
结果: <coroutine object say_hello at 0x000002C4727D12F0>

使用 await: Hello, Bob!


C:\Users\zying\AppData\Local\Temp\ipykernel_20952\145965430.py:17: RuntimeWarning: coroutine 'say_hello' was never awaited
  result = await say_hello("Bob")


## 5.3 asyncio 常用函数详解

### 函数速查表

| 函数 | 用途 | 返回值 | 是否需要 await |
|------|------|--------|----------------|
| `asyncio.sleep(n)` | 暂停 n 秒 | None | ✅ 是 |
| `asyncio.gather(*coros)` | 并发执行多个协程 | 结果列表 | ✅ 是 |
| `asyncio.create_task(coro)` | 创建并调度任务 | Task 对象 | ❌ 否 |
| `asyncio.wait(tasks)` | 等待任务完成 | (done, pending) | ✅ 是 |
| `asyncio.run(coro)` | 运行协程入口 | 协程返回值 | ❌ 否 |
| `asyncio.Queue()` | 异步队列 | Queue 对象 | ❌ 否 |

---

### asyncio.sleep()

异步睡眠，不会阻塞事件循环（与 `time.sleep()` 不同）：

```python
import asyncio

async def demo_sleep():
    print("开始睡眠...")
    await asyncio.sleep(2)  # 非阻塞等待
    print("醒来！")

# vs time.sleep() 会阻塞整个线程
def bad_sleep():
    print("开始睡眠...")
    time.sleep(2)  # 阻塞整个线程，其他协程无法运行
    print("醒来！")
```

---

### asyncio.gather()

并发执行多个协程，**收集所有结果**：

```python
import asyncio

async def fetch_user(user_id):
    await asyncio.sleep(0.5)
    return f"User{user_id}"

async def fetch_posts(user_id):
    await asyncio.sleep(0.3)
    return [f"Post1 by User{user_id}", f"Post2 by User{user_id}"]

async def main():
    # 并发执行，等待所有完成
    results = await asyncio.gather(
        fetch_user(1),
        fetch_posts(1),
        fetch_user(2),  # 可以同时运行3个任务
    )
    print(results)
    # 输出: ['User1', ['Post1 by User1', 'Post2 by User1'], 'User2']
```

**gather 高级用法**：

```python
async def task(name, fail=False):
    await asyncio.sleep(0.1)
    if fail:
        raise ValueError(f"{name} 失败了")
    return f"{name} 完成"

# 处理异常
results = await asyncio.gather(
    task("A"),
    task("B", fail=True),  # 这个会失败
    task("C"),
    return_exceptions=True  # 不抛出异常，返回异常对象
)
print(results)
# ['A 完成', ValueError('B 失败了'), 'C 完成']
```

---

### asyncio.create_task()

立即调度协程执行，**不等待完成**：

```python
import asyncio

async def background_task():
    print("后台任务开始")
    await asyncio.sleep(2)
    print("后台任务完成")
    return "后台结果"

async def main():
    # 创建任务，立即开始执行（不等待）
    task = asyncio.create_task(background_task())
    print("任务已创建，继续做其他事...")
    
    # 在任务运行时做其他工作
    await asyncio.sleep(1)
    print("主流程继续...")
    
    # 需要结果时再等待
    result = await task
    print(f"获得结果: {result}")
```

**gather vs create_task**：

| 场景 | 使用 |
|------|------|
| 需要所有结果 | `gather()` |
| 不需要立即等待 | `create_task()` |
| 后台执行任务 | `create_task()` |
| 批量并发 | `gather()` |

---

### asyncio.wait()

更灵活的等待方式：

```python
import asyncio

async def task(name, delay):
    await asyncio.sleep(delay)
    return f"{name} 完成"

async def main():
    tasks = [
        asyncio.create_task(task("A", 1)),
        asyncio.create_task(task("B", 2)),
        asyncio.create_task(task("C", 0.5)),
    ]
    
    # 等待第一个完成
    done, pending = await asyncio.wait(
        tasks, 
        return_when=asyncio.FIRST_COMPLETED
    )
    print(f"第一个完成: {[t.result() for t in done]}")
    
    # 取消剩余任务
    for t in pending:
        t.cancel()
```

---

### asyncio.Queue()

线程安全的异步队列，用于协程间通信：

```python
import asyncio

async def producer(queue):
    for i in range(5):
        await asyncio.sleep(0.1)
        await queue.put(f"Item {i}")
        print(f"生产: Item {i}")
    await queue.put(None)  # 发送结束信号

async def consumer(queue):
    while True:
        item = await queue.get()
        if item is None:
            break
        print(f"消费: {item}")
        await asyncio.sleep(0.2)
        queue.task_done()

async def main():
    queue = asyncio.Queue()
    await asyncio.gather(
        producer(queue),
        consumer(queue),
    )
```

---

### asyncio.run()

运行协程的入口（顶层调用）：

```python
# 在 .py 脚本中
import asyncio

async def main():
    print("Hello")

# 标准入口
if __name__ == "__main__":
    asyncio.run(main())

# ❌ 错误：不能在已有事件循环时调用
# ✅ Jupyter 中直接用 await
# ✅ 脚本中用 asyncio.run()
```

### 练习：并发获取多个数据

In [4]:
import asyncio

async def fetch_user(user_id):
    """模拟获取用户信息"""
    await asyncio.sleep(0.5)  # 模拟网络延迟
    return {"id": user_id, "name": f"User{user_id}"}

async def fetch_posts(user_id):
    """模拟获取用户文章"""
    await asyncio.sleep(0.3)
    return [
        {"id": 1, "title": f"Post by User{user_id}"},
        {"id": 2, "title": f"Another post by User{user_id}"},
    ]

async def fetch_comments(post_id):
    """模拟获取文章评论"""
    await asyncio.sleep(0.2)
    return [
        {"id": 1, "text": "Great post!"},
        {"id": 2, "text": "Thanks!"},
    ]

async def get_user_data(user_id):
    """获取用户完整数据（并发）"""
    start = asyncio.get_event_loop().time()
    
    # 并发获取用户信息和文章
    user, posts = await asyncio.gather(
        fetch_user(user_id),
        fetch_posts(user_id),
    )
    
    # 并发获取所有文章的评论
    comment_tasks = [fetch_comments(post["id"]) for post in posts]
    comments = await asyncio.gather(*comment_tasks)
    
    elapsed = asyncio.get_event_loop().time() - start
    
    return {
        "user": user,
        "posts": posts,
        "comments": comments,
        "time_elapsed": f"{elapsed:.2f}s",
    }

# 运行
result = await get_user_data(1)
print("用户数据:")
print(f"  用户: {result['user']}")
print(f"  文章数: {len(result['posts'])}")
print(f"  评论组数: {len(result['comments'])}")
print(f"  耗时: {result['time_elapsed']}")

用户数据:
  用户: {'id': 1, 'name': 'User1'}
  文章数: 2
  评论组数: 2
  耗时: 0.71s


## 5.4 并发 vs 并行

### 核心区别

这两个概念经常被混淆，但它们有本质的区别：

#### 并发 (Concurrency)

**同时处理多个任务**，但在任何时刻只执行一个任务。

```
时间线: ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
任务A:   ██        ██        ██
任务B:      ██        ██        ██
任务C:         ██        ██        ██

特点：交替执行，协作式切换
比喻：一个人同时处理多件事情（一边等水开，一边切菜）
```

**关键点**：
- 同一时间段内交替执行
- 适合 I/O 密集型任务（网络请求、文件读写）
- 单核 CPU 也能实现
- Python 通过 asyncio 实现

#### 并行 (Parallelism)

**同时执行多个任务**，在同一时刻真正同时运行。

```
时间线: ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
任务A:   ████████████████████
任务B:   ████████████████████
任务C:   ████████████████████
        ↑↑^ 三核同时执行

特点：同时执行，真正的同时
比喻：三个人同时各自处理一件事
```

**关键点**：
- 同一时刻真正同时执行
- 适合 CPU 密集型任务（图像处理、机器学习）
- 需要多核 CPU
- Python 通过 multiprocessing 实现

---

### 选择指南

| 任务类型 | 特征 | 推荐方案 | 原因 |
|----------|------|----------|------|
| **LLM API 调用** | 等待网络响应 | asyncio | I/O 等待时间远长于处理时间 |
| **数据库查询** | 等待数据库 | asyncio | 大量时间花在网络传输 |
| **向量搜索** | 计算 + I/O | asyncio | 现代向量库是 I/O 密集型 |
| **图像处理** | CPU 计算 | multiprocessing | 需要 CPU 并行计算 |
| **机器学习推理** | CPU/GPU 密集 | multiprocessing/torch | 充分利用多核 |
| **简单数据转换** | 少量 CPU | threading | 简单，开销小 |

---

### 代码对比

```python
import asyncio
import time
from concurrent.futures import ProcessPoolExecutor

# === 并发（asyncio）===
async def async_task(name):
    print(f"{name}: 开始")
    await asyncio.sleep(1)  # I/O 等待
    print(f"{name}: 完成")
    return name

async def concurrent_example():
    start = time.time()
    await asyncio.gather(
        async_task("A1"),
        async_task("A2"),
        async_task("A3"),
    )
    print(f"并发耗时: {time.time() - start:.2f}s\n")

# === 并行（multiprocessing）===
def cpu_task(name):
    print(f"{name}: 开始")
    sum(i * i for i in range(10_000_000))  # CPU 计算
    print(f"{name}: 完成")
    return name

def parallel_example():
    start = time.time()
    with ProcessPoolExecutor(max_workers=3) as executor:
        executor.map(cpu_task, ["B1", "B2", "B3"])
    print(f"并行耗时: {time.time() - start:.2f}s")
```

---

### Python 中的并发方式

| 方式 | 模块 | 适用场景 | GIL 限制 |
|------|------|----------|----------|
| **异步** | asyncio | I/O 密集型 | ✅ 不受限制 |
| **多线程** | threading | 简单 I/O | ❌ 受 GIL 限制 |
| **多进程** | multiprocessing | CPU 密集型 | ✅ 不受限制 |

**什么是 GIL？**

GIL (Global Interpreter Lock) 是 Python 解释器的全局锁，它确保同一时刻只有一个线程执行 Python 字节码。

- **asyncio** 避开 GIL：单线程，不需要锁
- **threading** 受 GIL 影响：多线程不能真正并行
- **multiprocessing** 绕过 GIL：每个进程有独立解释器

---

### Agent 开发中的选择

在 Agent 应用中，几乎总是选择 **asyncio**：

```python
# 典型的 Agent 异步流程
async def agent_process(user_query):
    # 1. 并发获取上下文
    history, context = await asyncio.gather(
        get_chat_history(user_id),     # 数据库查询
        search_knowledge(user_query),  # 向量搜索
    )
    
    # 2. 调用 LLM
    response = await llm.chat(user_query, context)
    
    # 3. 如果需要工具调用，并发执行工具
    if response.tool_calls:
        tool_results = await asyncio.gather(*[
            execute_tool(call) for call in response.tool_calls
        ])
    
    return response
```

为什么用异步：
- LLM API 调用慢（秒级）
- 向量数据库查询慢（毫秒到秒级）
- 网络请求慢
- 需要同时服务多个用户
- FastAPI 等框架原生支持异步

In [5]:
from backend.llm.base import BaseLLM
import inspect

print("=== BaseLLM 的异步方法 ===\n")

# 查看所有异步方法
for name, method in inspect.getmembers(BaseLLM, predicate=inspect.isfunction):
    if inspect.iscoroutinefunction(method):
        sig = inspect.signature(method)
        print(f"async {name}{sig}")

=== BaseLLM 的异步方法 ===

async chat(self, messages: List[backend.llm.models.Message], tools: List[backend.llm.models.ToolDefinition] | None = None, temperature: float = 0.7, max_tokens: int | None = None, **kwargs) -> backend.llm.models.ChatResponse
async embed(self, texts: List[str], **kwargs) -> backend.llm.models.EmbeddingResponse


## 5.6 异步上下文管理器

### 异步 with 语句

In [6]:
class AsyncConnection:
    """模拟异步连接管理器"""
    
    async def __aenter__(self):
        print("连接: 正在建立...")
        await asyncio.sleep(0.1)
        print("连接: 已建立")
        return self
    
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("连接: 正在关闭...")
        await asyncio.sleep(0.1)
        print("连接: 已关闭")
        return False
    
    async def query(self, sql):
        print(f"查询: {sql}")
        await asyncio.sleep(0.2)
        return f"{sql}的结果"

# 使用异步上下文管理器
async def use_connection():
    async with AsyncConnection() as conn:
        result = await conn.query("SELECT * FROM users")
        print(f"\n查询结果: {result}")
    # 退出时自动调用 __aexit__

await use_connection()

连接: 正在建立...
连接: 已建立
查询: SELECT * FROM users

查询结果: SELECT * FROM users的结果
连接: 正在关闭...
连接: 已关闭


## 5.7 异步迭代器

In [7]:
class AsyncStream:
    """模拟异步数据流"""
    
    def __init__(self, data):
        self.data = data
        self.index = 0
    
    def __aiter__(self):
        return self
    
    async def __anext__(self):
        await asyncio.sleep(0.1)  # 模拟数据到达延迟
        if self.index >= len(self.data):
            raise StopAsyncIteration
        value = self.data[self.index]
        self.index += 1
        return value

# 使用异步迭代器
async def process_stream():
    stream = AsyncStream(["数据1", "数据2", "数据3", "数据4"])
    
    print("异步迭代数据流:")
    async for item in stream:
        print(f"  处理: {item}")

await process_stream()

异步迭代数据流:
  处理: 数据1
  处理: 数据2
  处理: 数据3
  处理: 数据4


## 5.8 实战：模拟并发 LLM 调用

In [8]:
import asyncio
from pathlib import Path
from backend.llm import LLMFactory, Message

def get_deepseek_key():
    project_path = Path(os.getcwd())
    for path in [project_path] + list(project_path.parents):
        env_file = path / ".env"
        if env_file.exists():
            with open(env_file, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line.startswith("DEEPSEEK_API_KEY="):
                        return line.split("=", 1)[1].strip()
    return ""

async def ask_llm(question, index):
    """向 LLM 提问"""
    print(f"[任务{index}] 发送: {question}")
    
    api_key = get_deepseek_key()
    llm = LLMFactory.create("deepseek", api_key=api_key)
    
    start = asyncio.get_event_loop().time()
    response = await llm.chat([
        Message(role="user", content=question)
    ])
    elapsed = asyncio.get_event_loop().time() - start
    
    print(f"[任务{index}] 收到回答 (耗时{elapsed:.2f}s)")
    return {
        "question": question,
        "answer": response.content[:100] + "...",
        "time": elapsed,
    }

async def concurrent_questions():
    """并发提问多个问题"""
    questions = [
        "什么是Python？",
        "什么是异步编程？",
        "什么是LLM？",
    ]
    
    print("=== 并发提问 ===\n")
    start = asyncio.get_event_loop().time()
    
    # 并发执行
    results = await asyncio.gather(*[
        ask_llm(q, i+1) for i, q in enumerate(questions)
    ])
    
    total_time = asyncio.get_event_loop().time() - start
    
    print(f"\n=== 完成 ===")
    print(f"总耗时: {total_time:.2f}秒")
    print(f"平均耗时: {total_time/len(questions):.2f}秒/问题")
    
    return results

# 运行（取消注释以实际执行）
# await concurrent_questions()

## 5.9 常见陷阱与最佳实践

### 陷阱 1: 阻塞操作阻塞整个事件循环

```python
# ❌ 错误：time.sleep 会阻塞整个事件循环
import time

async def bad_example():
    print("开始")
    time.sleep(5)  # 阻塞！所有协程都被暂停
    print("结束")

# ✅ 正确：使用 asyncio.sleep
async def good_example():
    print("开始")
    await asyncio.sleep(5)  # 非阻塞，其他协程可以运行
    print("结束")
```

**其他阻塞操作**：
```python
# ❌ 阻塞操作
def bad_operations():
    time.sleep(1)           # 阻塞睡眠
    large_data = urllib.request(url).read()  # 阻塞网络
    with open(file) as f: content = f.read()  # 阻塞 I/O

# ✅ 使用异步库
async def good_operations():
    await asyncio.sleep(1)           # 异步睡眠
    async with aiohttp.ClientSession() as session:  # 异步网络
        async with session.get(url) as resp:
            data = await resp.read()
    async with aiofiles.open(file) as f:  # 异步文件
        content = await f.read()
```

---

### 陷阱 2: 忘记 await

```python
# ❌ 错误：调用协程但没有 await
async def bad():
    result = some_async_function()  # 返回协程对象，不是结果
    print(result)  # <coroutine object...>

# ✅ 正确
async def good():
    result = await some_async_function()
    print(result)  # 实际结果
```

---

### 陷阱 3: 在普通函数中调用 async 函数

```python
# ❌ 错误
def sync_function():
    result = await async_function()  # SyntaxError

# ✅ 方案1: 把调用者也改为 async
async def async_function():
    result = await async_function()

# ✅ 方案2: 使用 asyncio.run（仅顶层）
def sync_function():
    result = asyncio.run(async_function())

# ✅ 方案3: 在已有事件循环中使用 create_task
def sync_function():
    loop = asyncio.get_event_loop()
    task = loop.create_task(async_function())
    # task 现在在后台运行
```

---

### 陷阱 4: asyncio.run() 嵌套调用

```python
# ❌ 错误：不能在已有事件循环中调用 asyncio.run()
async def main():
    result = asyncio.run(other_async())  # RuntimeError!

# ✅ 正确：直接使用 await
async def main():
    result = await other_async()
```

---

### 陷阱 5: 任务未等待完成

```python
# ⚠️ 潜在问题：任务可能永远不会执行
async def bad():
    task = asyncio.create_task(some_coro())
    # 函数结束，任务可能还没开始执行
    return

# ✅ 正确
async def good():
    task = asyncio.create_task(some_coro())
    result = await task  # 等待任务完成
    return result
```

---

### 陷阱 6: 异常被吞掉

```python
# ❌ 错误：create_task 的异常不会被传播
async def bad():
    task = asyncio.create_task(will_fail())
    # 如果 await task，异常会被抛出
    # 如果不 await，异常会被打印但不影响程序

# ✅ 使用 try-except
async def good():
    try:
        await will_fail()
    except Exception as e:
        print(f"捕获异常: {e}")

# ✅ 或使用 gather 的异常处理
async def better():
    results = await asyncio.gather(
        task1(),
        task2(),
        return_exceptions=True  # 异常作为结果返回
    )
    for result in results:
        if isinstance(result, Exception):
            print(f"任务失败: {result}")
```

---

### 最佳实践

#### 1. 使用类型提示

```python
from typing import List

async def fetch_users() -> List[dict]:
    """获取用户列表"""
    ...
```

#### 2. 为协程函数命名

```python
# 好的命名习惯
async def fetch_user(user_id: int) -> dict: ...
async def save_user(user: dict) -> bool: ...
async def process_data(data: list) -> None: ...
```

#### 3. 合理设置超时

```python
async def safe_call():
    try:
        result = await asyncio.wait_for(
            risky_operation(), 
            timeout=5.0  # 最多等待5秒
        )
    except asyncio.TimeoutError:
        print("操作超时")
        # 处理超时
```

#### 4. 使用 Semaphore 限制并发

```python
# 避免同时发起太多请求
sem = asyncio.Semaphore(10)  # 最多10个并发

async def limited_request(url):
    async with sem:  # 获取信号量
        return await fetch(url)

# 使用
await asyncio.gather(*[
    limited_request(url) for url in urls  # 最多10个并发
])
```

#### 5. 正确使用 asyncio.shield

```python
# 保护任务不被取消
async def main():
    task = asyncio.create_task(important_work())
    
    try:
        await asyncio.wait_for(some_other_work(), timeout=1)
    except asyncio.TimeoutError:
        # 重要任务继续运行
        await task  # 等待重要任务完成
```

---

### 常见面试问题

**Q: 异步和线程有什么区别？**

| 特性 | asyncio (协程) | threading (线程) |
|------|----------------|------------------|
| **调度方式** | 协作式（自己让出） | 抢占式（OS 切换） |
| **内存开销** | 极低（KB 级） | 较高（MB 级） |
| **创建成本** | 几乎为 0 | 较高 |
| **GIL 限制** | 不受影响 | 受 GIL 限制 |
| **适用场景** | I/O 密集型 | I/O 或简单 CPU |
| **调试难度** | 较高 | 较低 |
| **上下文切换** | 快（用户态） | 慢（内核态） |

**Q: 什么是事件循环？**

事件循环是 asyncio 的核心，负责：
1. 注册和执行协程
2. 管理任务调度
3. 处理 I/O 事件（网络、文件）
4. 管理定时器和回调

```python
# 事件循环工作流程
# 1. 注册任务
loop.create_task(coro1)
loop.create_task(coro2)

# 2. 事件循环运行
# - 当 coro1 await 时，切换到 coro2
# - 当 I/O 就绪时，恢复等待的协程
# - 当协程完成时，获取结果
```

**Q: await 后面可以接什么？**

1. **协程函数调用**：`await async_func()`
2. **协程对象**：`await coroutine_object`
3. **Task 对象**：`await asyncio.create_task(coro)`
4. **Future 对象**：`await future`
5. **实现了 `__await__` 的对象**

**Q: asyncio.gather() 和 asyncio.create_task() 有什么区别？**

| | gather() | create_task() |
|---|----------|---------------|
| **返回** | 结果列表 | Task 对象 |
| **等待** | 等待所有完成 | 立即返回，不等待 |
| **用途** | 需要所有结果 | 后台执行、不立即等待 |
| **异常** | 第一个异常立即抛出 | 需要手动 await 才能看到异常 |

**Q: 如何取消正在运行的异步任务？**

```python
async def cancel_example():
    # 创建任务
    task = asyncio.create_task(long_running_task())
    
    # 等待一点时间
    await asyncio.sleep(1)
    
    # 取消任务
    task.cancel()
    
    try:
        await task  # 会触发 CancelledError
    except asyncio.CancelledError:
        print("任务被取消")
```

---

## 总结

### 核心知识回顾

| 知识点 | 说明 | 关键点 |
|--------|------|--------|
| **async/await** | 定义和调用协程 | `async def` 定义，`await` 调用 |
| **协程 (Coroutine)** | 可暂停恢复的函数 | 调用返回协程对象，不立即执行 |
| **事件循环** | 协程调度器 | 管理协程的执行和切换 |
| **asyncio** | 异步 I/O 标准库 | 提供 sleep、gather、create_task 等 |
| **并发 vs 并行** | 概念区别 | 并发=交替执行，并行=同时执行 |
| **async with** | 异步上下文管理器 | 用于资源管理 |
| **async for** | 异步迭代器 | 用于异步数据流 |

### 异步编程的黄金法则

1. **I/O 密集型用异步，CPU 密集型用多进程**
2. **永远不要在 async 函数中使用阻塞操作**（time.sleep、urllib 等）
3. **await 不能省略**，否则协程不会执行
4. **asyncio.run() 只能在顶层调用一次**
5. **合理控制并发数**，用 Semaphore 限制
6. **设置超时**，避免无限等待
7. **处理异常**，gather 使用 return_exceptions=True

### 在 Agent 开发中的应用

```
┌─────────────────────────────────────────────────────┐
│              异步 Agent 的典型流程                    │
├─────────────────────────────────────────────────────┤
│  1. 用户请求 → FastAPI (异步框架)                    │
│  2. Agent.run() (async)                              │
│  3. 并发获取: history + context (asyncio.gather)    │
│  4. 调用 LLM API (await llm.chat)                   │
│  5. 如需工具: 并发执行所有工具 (asyncio.gather)      │
│  6. 保存记忆 (await memory.save)                    │
│  7. 返回结果                                         │
└─────────────────────────────────────────────────────┘
```

### 进一步学习

- **官方文档**: https://docs.python.org/3/library/asyncio.html
- **异步库推荐**:
  - `aiohttp` - 异步 HTTP 客户端
  - `aiofiles` - 异步文件操作
  - `asyncpg` - 异步 PostgreSQL
  - `motor` - 异步 MongoDB

### 记住

> 异步编程是 Agent 开发的必备技能，因为它能显著提升 I/O 密集型应用的性能。掌握 async/await 和 asyncio，你就能构建高效的 AI 应用！